## U.S. Employer Health Insurance: An Exploratory Data Analysis using MEPS-IC Survey Data

## Background

This project uses data from the **Medical Expenditure Panel Survey – Insurance Component (MEPS-IC)**, published by the Agency for Healthcare Research and Quality (AHRQ).

It's an annual survey that goes back to 1996 and collects data directly from private-sector employers on things like how much they spend on health insurance, what plans they offer, and how many employees actually enroll. I chose to focus only on the private sector data.

**Source:** [https://datatools.ahrq.gov/meps-ic/](https://datatools.ahrq.gov/meps-ic/)

---

## Why I chose this dataset

As part of the Financial, Actuarial and Analytics team at WTW, I spend most of my time analyzing cost and utilisation trends to help clients make decisions about their benefits strategy. A big part of that work is benchmarking, helping clients understand whether what they're spending is reasonable for their size and industry. But since I work in a GDC (Global Delivery Centre), this data mostly sits with the consulting teams who directly deal with clients. To understand the broader context my work feeds into, I looked for a public dataset that covers similar ground. The MEPS-IC survey made sense since it tracks the same things I work with day to day (enrollment, premiums, employer contributions, etc.) but just at a national level.

---

## Questions I want to answer

1. How have average health insurance premiums changed over time?
2. Does firm size affect how much employees pay?
3. Which industries have the highest premiums?
4. How is the premium split between employer and employee?
5. How have employee enrollment rates changed over time?

## Loading the Data

In [ ]:
# Importing libraries I will need
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

In [ ]:
# Loading the cleaned Excel file into a dataframe
df = pd.read_excel("MEPSIC_Survey_Data.xlsx")
print("Data loaded successfully.")

In [ ]:
# Performing a quick check
print("Total rows:", len(df))
print("Total columns:", len(df.columns))
df.head()

## Exploring the Data

Before I write any queries, I want to understand what fields we have and the values contained within them. The MEPS-IC data isn't structured like a typical table, but instead, organized in a long-format table of different estimates with different groupings. 

In [ ]:
# Checking columns and data types
df.info()

In [ ]:
# Loading the data schema to check what data each column represents
schema_df = pd.read_excel("MEPSIC_Survey_Data_Schema.xlsx")
pd.set_option('display.max_colwidth', None)
display(schema_df)

In [ ]:
# Filtering the schema to only show columns relevant to our analysis
columns_we_use = ['year', 'estimate', 'estimate_category', 'row_group','row_label', 'col_group', 'col_label', 'value_rounded']
filtered_schema = schema_df[schema_df['Column Name'].isin(columns_we_use)]
display(filtered_schema[['Column Name', 'Description']])

In [ ]:
# The estimate column is our key field - capping at 25 since there are a lot of unique values
df['estimate'].value_counts().head(25)

In [ ]:
# Checking what estimate_category, estimate_subcategory, and row_group look like
display("estimate_category values:")
display(df['estimate_category'].unique())
print()
display("estimate_subcategory values:")
display(df['estimate_subcategory'].unique())
print()
display("row_group values:")
display(df['row_group'].unique())

In [ ]:
# Also checking col_group and col_label
display("col_group values:")
display(df['col_group'].unique())
print()
display("col_label values:")
display(df['col_label'].unique())

In [ ]:
# Checking for any null or duplicate values
display("Null counts per column:")
display(df.isnull().sum())
print()
display("Duplicate rows:")
display(df.duplicated().sum(), "duplicate rows found")
print()
display("Value range for value_rounded:")
display(df['value_rounded'].describe())

In [ ]:
# Loading the dataframe into an in-memory SQLite database so I can run queries directly inside the notebook.
engine = create_engine("sqlite:///:memory:")
df.to_sql("meps_ic", con=engine, index=False, if_exists="replace")

print("Database is ready.")

In [ ]:
# Adding a helper function to avoid redundancy
def run_query(sql):
    return pd.read_sql_query(sql, con=engine)

---

## Question 1: How have average single premiums changed over time?

Health insurance costs have been rising for years, but by how much exactly? I want to see the national average single coverage premium for each year available in the dataset.

In [ ]:
# Note: I had to use 'LIKE' instead of '=' for the estimate column because the strings have some extra text in them and an exact match returns nothing.
query = """
SELECT 
    year,
    CAST(ROUND(AVG(value_rounded), 0) AS INTEGER) AS avg_single_premium
FROM meps_ic
WHERE estimate LIKE '%Average total premium%'
  AND estimate_category = 'Single coverage'
  AND col_label = 'Total'
GROUP BY year
ORDER BY year
"""
premium_trend = run_query(query)
display(premium_trend)

In [ ]:
# Plotting a line chart to see the overall trend
plt.figure(figsize=(10, 5))
plt.plot(premium_trend['year'], premium_trend['avg_single_premium'], marker='o', color='#003f5c', linewidth=2)
plt.title('Average Single Health Insurance Premium Over Time')
plt.ylabel('Average Premium ($)')
plt.xlabel('Year')
plt.tight_layout()
plt.show()

**What I found:** The rise is pretty striking when you see it laid out like this. Premiums have gone up roughly 4x from 1996 to now, and there's no real dip, even during the 2008 recession it just keeps going up. This graph highlights why benchmarking is important; if premiums are rising faster than the national trend, it's worth looking into. 

---

## Question 2: Does company size affect how much employees pay in premiums?

My assumption going into this was that larger companies pay less because they have more bargaining power with insurers. I want to check if that's actually true for 2024.

In [ ]:
query = """
SELECT 
    col_label AS firm_size,
    CAST(ROUND(AVG(value_rounded), 0) AS INTEGER) AS avg_premium
FROM meps_ic
WHERE year = 2024
  AND estimate LIKE '%Average total premium%'
  AND estimate_category = 'Single coverage'
  AND col_label NOT IN ('Total', 'Less than 50 employees', '50 or more employees')
GROUP BY col_label
ORDER BY avg_premium DESC
"""
firm_size = run_query(query)
display(firm_size)

In [ ]:
# The query returns results sorted by premium (highest to lowest) but I want the bar chart sorted by firm size (smallest to largest)
size_order = ['Less than 10 employees', '10-24 employees',
              '25-99 employees', '100-999 employees', '1000 or more employees']
firm_size['firm_size'] = pd.Categorical(firm_size['firm_size'], categories=size_order, ordered=True)
firm_size = firm_size.sort_values('firm_size')

plt.figure(figsize=(10, 5))
plt.bar(firm_size['firm_size'], firm_size['avg_premium'], color='#003f5c')
plt.title('Average Single Premium by Firm Size (2024)')
plt.ylabel('Average Premium ($)')
plt.xlabel('Firm Size')
plt.tight_layout()
plt.show()

**What I found:** Interestingly, my assumptions did not hold as well as I had expected. The smallest firms do pay the highest average premium, but the gap between them and the largest firms is surprisingly small. I thought there'd be a much steeper drop as firm size grew. 

One possible explanation: large firms tend to offer richer plan designs, which keeps their total premium high even if they negotiate a better rate. This is something that has come up repeatedly in conversations with our consultants, based on feedback they’ve gathered while working closely with clients.

---

## Question 3: Which industries have the highest premiums?

Now I'm curious whether industry matters more than firm size. Some sectors might have older workforces, or just offer more generous benefits overall. Let me rank all the industries by their average single premium in 2024.

In [ ]:
query = """
SELECT 
    row_label AS industry,
    CAST(ROUND(AVG(value_rounded), 0) AS INTEGER) AS avg_premium
FROM meps_ic
WHERE year = 2024
  AND estimate LIKE '%Average total premium%'
  AND estimate_category = 'Single coverage'
  AND row_group = 'Industry group'
  AND col_label = 'Total'
GROUP BY row_label
ORDER BY avg_premium DESC
"""
by_industry = run_query(query)
display(by_industry)

In [ ]:
# Plotting a horizontal bar chart so the industry labels don't get cut off
plt.figure(figsize=(10, 6))
plt.barh(by_industry['industry'][::-1], by_industry['avg_premium'][::-1], color='#003f5c')
plt.title('Average Single Premium by Industry (2024)')
plt.xlabel('Average Premium ($)')
plt.tight_layout()
plt.show()

**What I found:** I expected Construction or Mining to top this list since those are high-risk industries, but instead, it's Professional Services and Finance/Insurance at the top. Although it makes sense since those industries compete hard on compensation and benefits to attract talent, so they offer richer plans. Agriculture and Construction being at the bottom also makes sense given how many workers in that sector are seasonal or work through subcontractors, so they may not be on employer plans at all.

---

## Question 4: How is the premium split between employer and employee?

Looking at the total premium is useful but I want to know who's actually paying it. Does the employer absorb more in larger firms? Let me compare the employer vs. employee share by firm size for 2024.

In [ ]:
# Calculating total premium and employee contribution
query = """
WITH total AS (
    SELECT 
        col_label AS firm_size,
        CAST(ROUND(AVG(value_rounded), 0) AS INTEGER) AS total_premium
    FROM meps_ic
    WHERE year = 2024
      AND estimate LIKE '%Average total premium%'
      AND estimate_category = 'Single coverage'
      AND col_label NOT IN ('Total', 'Less than 50 employees', '50 or more employees')
    GROUP BY col_label
),
contrib AS (
    SELECT 
        col_label AS firm_size,
        CAST(ROUND(AVG(value_rounded), 0) AS INTEGER) AS employee_contribution
    FROM meps_ic
    WHERE year = 2024
      AND estimate LIKE '%Average total employee contribution%'
      AND estimate_category = 'Single coverage'
      AND col_label NOT IN ('Total', 'Less than 50 employees', '50 or more employees')
    GROUP BY col_label
)
SELECT
    t.firm_size,
    t.total_premium,
    c.employee_contribution,
    t.total_premium - c.employee_contribution AS employer_contribution,
    ROUND(100.0 * c.employee_contribution / t.total_premium, 1) AS employee_pct,
    ROUND(100.0 * (t.total_premium - c.employee_contribution) / t.total_premium, 1) AS employer_pct
FROM total t
JOIN contrib c ON t.firm_size = c.firm_size
"""
split = run_query(query)
display(split)

In [ ]:
# Sorting by firm size
split['firm_size'] = pd.Categorical(split['firm_size'], categories=size_order, ordered=True)
split = split.sort_values('firm_size').reset_index(drop=True)

#Using subplots for text labels
fig, ax = plt.subplots(figsize=(12, 5))
bars_employer = ax.bar(split['firm_size'], split['employer_contribution'], label='Employer pays', color='#003f5c', width=0.8)
bars_employee = ax.bar(split['firm_size'], split['employee_contribution'], bottom=split['employer_contribution'], label='Employee pays', color='#006770', width=0.8)

# Using loop to add percentage labels
for idx, row in split.iterrows():
    ax.text(idx, row['employer_contribution'] / 2,
            f"{row['employer_pct']}%", ha='center', va='center', color='white', fontsize=10)
    ax.text(idx, row['employer_contribution'] + row['employee_contribution'] / 2,
            f"{row['employee_pct']}%", ha='center', va='center', color='white', fontsize=10)

ax.set_title('Premium Split: Employer vs Employee by Firm Size (2024)')
ax.set_ylabel('Premium ($)')
ax.set_ylim(0, 10000)
ax.legend()
plt.tight_layout()
plt.show()

**What I found:** As expected, employers consistently absorb the larger share of the premium across all firm sizes. What's interesting though is that smaller firms are paying the most in absolute dollar terms, which means they're getting hit twice: higher total premiums and a bigger contribution out of pocket.

---

## Question 5: How have employee enrollment rates changed over time?

Just because a company offers health insurance doesn't mean everyone signs up. I want to look at what percentage of employees actually enroll, and whether that's changed over time. Rather than looking at all industries at once, I'll focus on the top 5 by enrollment rate.

In [ ]:
# Using LIMIT 6 because 'Unknown' appears in the top results and is filtered out below
query_top5 = """
SELECT 
    row_label AS industry,
    ROUND(AVG(value_rounded), 1) AS avg_enrollment
FROM meps_ic
WHERE estimate LIKE '%Percent of employees enrolled%'
  AND row_group = 'Industry group'
  and row_label != 'Unknown'
  AND col_label = 'Total'
  AND year >= 2000
GROUP BY row_label
ORDER BY avg_enrollment DESC
LIMIT 5
"""
top5_df = run_query(query_top5)

# Removing 'Unknown' from our category list
top_5 = top5_df[top5_df['industry'] != 'Unknown']['industry'].tolist()
display(top_5)

In [ ]:
# Pulling the year-by-year enrollment rate for all industries, excluding 'Unknown'
query = """
SELECT 
    year,
    row_label AS industry,
    ROUND(AVG(value_rounded), 1) AS enrollment_rate
FROM meps_ic
WHERE estimate LIKE '%Percent of employees enrolled%'
  AND row_group = 'Industry group'
  AND col_label = 'Total'
  AND year >= 2000
GROUP BY year, row_label
ORDER BY year, industry
"""
enrollment_trend = run_query(query)

# Filtering the dataset to include only the top 5 industries
enrollment_trend = enrollment_trend[enrollment_trend['industry'].isin(top_5)]
display(enrollment_trend.head())

In [ ]:
#Plotting a multi-line time series chart to compare enrollment trends across industries
plt.figure(figsize=(12, 6))

colors = ['#003f5c', '#006770', '#008c54', '#7aa609', '#ffa600']

for i, industry in enumerate(top_5):
    data = enrollment_trend[enrollment_trend['industry'] == industry]
    plt.plot(data['year'], data['enrollment_rate'], marker='o', label=industry, color=colors[i])

# Adding a 60% reference line for comparison across industries
plt.axhline(y=60, color='gray', linestyle='--', label='60% line')
plt.title('Enrollment Rate Over Time: Top 5 Industries (2000–2024)')
plt.xlabel('Year')
plt.ylabel('Enrollment Rate (%)')
plt.xticks(range(2000, 2025, 4))
plt.legend(loc='upper right', bbox_to_anchor=(1, 0.95), fontsize=10)
plt.tight_layout()
plt.show()

**What I found:** Enrollment rates dropped around 2001, stayed flat between 30–40% for nearly two decades, then shot up after 2019, probably due to COVID. I was surprised the numbers never crossed 60%, so I looked into it. Turns out a lot of non-enrollees aren't actually uninsured; they're either on a spouse's plan or Medicaid. Cost seems to be the other big reason, especially for lower-wage workers. That would also explain why Agriculture and Construction sit at the bottom.

## Key Takeaways
- Premiums have risen consistently for nearly three decades with no meaningful plateaus, putting pressure on employers of all sizes.
- Firm size matters less than expected. Smaller firms pay the highest premiums, but larger firms are not far behind, likely because they offer richer plans.
- Industry and workforce composition are stronger predictors of both premium levels and enrollment rates than firm size alone.
- Enrollment improved after 2019, but rates still sit under 60% in most industries, meaning cost remains a real barrier for many workers.

**Bottom line:** It's pretty clear from the analysis that rising premiums are not a new problem. What's less obvious until you dig into it is how much industry and plan design shape the numbers, often more than firm size does. Benchmarking exists precisely because these patterns are easy to miss without the right context.